In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

NUM_WOUNDS = 500
DAYS = 14

data = []

# Ideal values
IDEAL_PH = 6.0
IDEAL_TEMP = 37.0
IDEAL_MOISTURE = 70

for w in range(NUM_WOUNDS):
    wound_id = f"W{w:03d}"
    infected = np.random.rand() < 0.3  # 30% infected

    for day in range(1, DAYS + 1):

        # Simulate sensor values
        pH = np.random.normal(6.8 if infected else 6.0, 0.3)
        temp = np.random.normal(38.2 if infected else 37.0, 0.4)
        moisture = np.random.normal(55 if infected else 70, 6)

        # -------------------------
        # Infection Risk (%)
        # -------------------------
        infection_risk = (
            (pH - 5.5) * 30 +
            (temp - 36.5) * 25 +
            (80 - moisture) * 0.4
        )
        infection_risk = np.clip(infection_risk + np.random.normal(0, 5), 0, 100)

        # -------------------------
        # Healing Progress (%)
        # -------------------------

        # Max healing allowed by day (timeline control)
        max_healing_by_day = (day / DAYS) * 100

        # Condition quality score (0–1)
        condition_penalty = (
            abs(pH - IDEAL_PH) * 0.2 +
            abs(temp - IDEAL_TEMP) * 0.3 +
            abs(moisture - IDEAL_MOISTURE) * 0.01
        )

        condition_score = np.clip(1 - condition_penalty, 0.2, 1.0)

        healing_progress = max_healing_by_day * condition_score
        healing_progress += np.random.normal(0, 3)

        healing_progress = np.clip(healing_progress, 0, max_healing_by_day)

        data.append([
            wound_id,
            day,
            round(pH, 2),
            round(temp, 2),
            round(moisture, 1),
            int(infection_risk),
            int(healing_progress)
        ])

df = pd.DataFrame(
    data,
    columns=[
        "wound_id",
        "day",
        "pH",
        "temperature",
        "moisture",
        "infection_risk_percent",
        "healing_progress_percent"
    ]
)

df.to_csv("wound_dataset_v2.csv", index=False)
print(df.head())


  wound_id  day    pH  temperature  moisture  infection_risk_percent  \
0     W000    1  5.67        37.13      71.7                      29   
1     W000    2  5.84        36.77      64.5                      10   
2     W000    3  6.24        36.39      67.4                      20   
3     W000    4  5.36        36.75      73.6                      17   
4     W000    5  6.04        36.79      66.4                      33   

   healing_progress_percent  
0                         4  
1                        14  
2                        13  
3                        22  
4                        32  


In [ ]:
import pandas as pd
import joblib
from xgboost import XGBRegressor

# Load dataset
df = pd.read_csv("wound_dataset_v2.csv")

# -------------------------
# FEATURES
# -------------------------
features = ["day", "pH", "temperature", "moisture"]

X = df[features]

# -------------------------
# TARGETS
# -------------------------
y_infection = df["infection_risk_percent"]
y_healing = df["healing_progress_percent"]

# -------------------------
# MODELS
# -------------------------
infection_model = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)

healing_model = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)

# -------------------------
# TRAIN
# -------------------------
infection_model.fit(X, y_infection)
healing_model.fit(X, y_healing)

# -------------------------
# SAVE
# -------------------------
joblib.dump(infection_model, "infection_model.pkl")
joblib.dump(healing_model, "healing_model.pkl")

print("Models trained and saved successfully.")


Models trained and saved successfully.


#Demo

In [ ]:
import joblib
import numpy as np

# -------------------------
# Load trained models
# -------------------------
infection_model = joblib.load("/content/drive/MyDrive/NanoHealAI/infection_model.pkl")
healing_model = joblib.load("/content/drive/MyDrive/NanoHealAI/healing_model.pkl")

# -------------------------
# Recommendation logic
# -------------------------
def get_recommendation(infection, healing):
    if infection > 70:
        return (
            "High risk – seek medical attention",
            [
                "Keep wound clean",
                "Seek professional medical care",
                "Change dressing as advised"
            ],
            [
                "Do not self-medicate",
                "Do not ignore pain or discharge",
                "Do not keep wound overly dry or wet"
            ]
        )
    elif infection > 40:
        return (
            "Moderate risk – monitor closely",
            [
                "Change dressing regularly",
                "Maintain proper hygiene",
                "Monitor wound daily"
            ],
            [
                "Avoid touching wound unnecessarily",
                "Avoid exposure to dirt or water",
                "Do not delay care if symptoms worsen"
            ]
        )
    else:
        return (
            "Low risk – healing normally",
            [
                "Maintain hygiene",
                "Follow dressing schedule",
                "Allow adequate airflow"
            ],
            [
                "Avoid excessive pressure",
                "Do not skip dressing changes",
                "Avoid harsh chemicals on wound"
            ]
        )

# -------------------------
# Bandage description (≤10 words)
# -------------------------
def bandage_status(healing):
    if healing < 30:
        return "Protecting wound and monitoring early healing"
    elif healing < 70:
        return "Maintaining moisture and supporting tissue regeneration"
    else:
        return "Supporting final healing and skin restoration"

# -------------------------
# USER INPUT (LIVE DEMO)
# -------------------------
pH = float(input("Enter wound pH: "))
temp = float(input("Enter temperature (°C): "))
moisture = float(input("Enter moisture (%): "))
day = int(input("Enter day: "))

# -------------------------
# Prediction
# -------------------------
X = np.array([[day, pH, temp, moisture]])

infection_risk = infection_model.predict(X)[0]
healing_progress = healing_model.predict(X)[0]

# Enforce valid medical bounds
infection_risk = int(np.clip(infection_risk, 0, 100))
healing_progress = int(np.clip(healing_progress, 0, 100))

recommendation, dos, donts = get_recommendation(infection_risk, healing_progress)
bandage_info = bandage_status(healing_progress)

# -------------------------
# OUTPUT (JUDGE FRIENDLY)
# -------------------------
print("\n--- Wound Assessment ---")
print(f"Day: {day}")
print(f"Infection Risk: {infection_risk}%")
print(f"Healing Progress: {healing_progress}%")
print(f"Bandage Status: {bandage_info}")
print(f"Recommendation: {recommendation}")

print("\nDo's:")
for d in dos:
    print(f"- {d}")

print("\nDon'ts:")
for d in donts:
    print(f"- {d}")


Enter wound pH: 5.34
Enter temperature (°C): 29.6
Enter moisture (%): 10.2
Enter day: 9

--- Wound Assessment ---
Day: 9
Infection Risk: 9%
Healing Progress: 19%
Bandage Status: Protecting wound and monitoring early healing
Recommendation: Low risk – healing normally

Do's:
- Maintain hygiene
- Follow dressing schedule
- Allow adequate airflow

Don'ts:
- Avoid excessive pressure
- Do not skip dressing changes
- Avoid harsh chemicals on wound


In [ ]:
import numpy as np

# Randomize realistic wound values
day = np.random.randint(1, 15)                       # Day 1 to 14
pH = round(np.random.uniform(5.5, 8.0), 2)           # Typical wound pH
temperature = round(np.random.uniform(36.0, 39.0), 2) # °C
moisture = round(np.random.uniform(30, 80), 1)       # % moisture

# Display the randomized values
print(f"Randomized values for demo:")
print(f"pH = {pH}")
print(f"Temperature = {temperature} °C")
print(f"Moisture = {moisture} %")
print(f"Day = {day}")


Randomized values for demo:
pH = 5.83
Temperature = 38.02 °C
Moisture = 71.2 %
Day = 7
